Do laboratório à equação diferencial: uma sensibilização sobre significados

In [ ]:
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import ipywidgets as widgets


type Listener = Callable[[Any], None]
"""Tipo de um listener: função que recebe o novo valor de um nó."""

type Transform = Callable[..., Any]
"""Tipo de uma transformação aplicada a valores de nós."""

type Number = int | float

In [ ]:
"""
Sistema reativo mínimo para notebooks interativos
==================================================

Este módulo implementa um pequeno grafo reativo baseado em **nós** (*nodes*)
e **listeners**.  O objetivo é permitir que widgets do :mod:`ipywidgets`
reajam automaticamente a mudanças de valor, sem acoplamento direto entre
produtores e consumidores.

Conceito central — listeners
-----------------------------
Um *listener* é uma função com a assinatura ``(value: Any) -> None`` que é
*registrada* em um :class:`ReactiveNode` via :meth:`ReactiveNode.subscribe`.
Sempre que :meth:`ReactiveNode.set` é chamado, o nó itera sobre todos os
listeners cadastrados e os invoca com o novo valor.

A maneira padrão de tornar um widget reativo é envolver a atualização
desejada do widget em uma função e registrá-la como listener::

    node = ReactiveNode(0)
    label = widgets.Label()

    # Registra um listener que mantém o texto do label sincronizado
    node.subscribe(lambda v: setattr(label, "value", str(v)))

    node.set(42)   # dispara o listener → label exibe "42"

O método de conveniência :meth:`ReactiveNode.to_widget` encapsula exatamente
esse padrão, eliminando o :func:`setattr` explícito::

    node.to_widget(label, "value", transform=str)

Para criar nós derivados que reagem a outros nós (compondo o grafo),
utilize :meth:`ReactiveNode.map` (um nó de entrada) ou :func:`computed`
(múltiplos nós de entrada).

Fluxo resumido
--------------
::

    WidgetNode  ──set()──►  ReactiveNode  ──set()──►  ReactiveNode
    (fonte)                 (derivado via map/computed) (saída → widget)
                                 │
                            listeners notificados
                            em cadeia

Classes e funções principais
-----------------------------
- :class:`ReactiveNode` — nó base com suporte a listeners.
- :class:`WidgetNode`   — nó reativo cuja *fonte* é um widget observável.
- :func:`bind`          — atalho para criar um :class:`WidgetNode`.
- :func:`computed`      — nó derivado de múltiplas dependências.
"""


def identity(x: Any) -> Any:
    """Retorna o argumento sem modificações.

    Usada como transformação padrão em :meth:`ReactiveNode.to_widget`.

    Parameters
    ----------
    x : Any
        Valor a ser retornado.

    Returns
    -------
    Any
        O mesmo objeto recebido.
    """
    return x


@dataclass(slots=True)
class ReactiveNode:
    """
    Nó reativo que armazena um valor e notifica listeners quando esse valor muda.

    Esta classe é o bloco fundamental do sistema reativo.  Um nó pode
    desempenhar três papéis:

    - **fonte primária** — alimentado externamente (e.g. por um :class:`WidgetNode`);
    - **valor derivado** — calculado a partir de outros nós via :meth:`map` ou
      :func:`computed`;
    - **produtor de saída** — direciona seu valor a um widget via
      :meth:`to_widget`.

    Mecanismo de listeners
    ----------------------
    Um *listener* é qualquer callable ``(value: Any) -> None`` registrado com
    :meth:`subscribe`.  A lista de listeners é mantida em :attr:`listeners`.

    Toda vez que :meth:`set` é chamado:

    1. O atributo :attr:`value` é atualizado.
    2. :meth:`_notify` percorre :attr:`listeners` e invoca cada um com o novo
       valor.

    Isso cria uma cadeia de propagação: alterar um nó raiz dispara, em cascata,
    todos os nós e widgets que dependem dele.

    Para tornar um widget reativo, basta criar uma função que o atualize e
    registrá-la como listener::

        node = ReactiveNode(0)
        slider_out = widgets.IntSlider()

        node.subscribe(lambda v: setattr(slider_out, "value", int(v)))

        node.set(7)   # slider_out.value passa a ser 7

    O método :meth:`to_widget` encapsula esse padrão de forma declarativa.

    Parameters
    ----------
    value : Any, optional
        Valor inicial do nó.  Padrão: ``None``.
    listeners : list[Listener], optional
        Lista de callbacks pré-registrados.  Em geral não é fornecida
        diretamente; use :meth:`subscribe` para registrar listeners após a
        criação.

    Examples
    --------
    Nó simples com listener impresso no console:

    >>> node = ReactiveNode(0)
    >>> node.subscribe(lambda v: print(f"novo valor: {v}"))
    ReactiveNode(value=0, listeners=[...])
    >>> node.set(3)
    novo valor: 3

    Ver também
    ----------
    WidgetNode : nó reativo cuja fonte é um widget do ipywidgets.
    computed   : cria um nó derivado de múltiplas dependências.
    """

    value: Any = None
    listeners: list[Listener] = field(default_factory=list)

    def get(self) -> Any:
        """
        Devolve o valor atual do nó.

        Returns
        -------
        Any
            Valor atualmente armazenado em :attr:`value`.
        """
        return self.value

    def set(self, value: Any) -> None:
        """
        Atualiza o valor do nó e notifica todos os listeners registrados.

        A notificação é síncrona e segue a ordem de registro dos listeners.
        Listeners adicionados durante a notificação *não* serão chamados na
        rodada atual (a lista é copiada antes da iteração).

        Parameters
        ----------
        value : Any
            Novo valor do nó.

        See Also
        --------
        subscribe : registra um listener para ser chamado por este método.
        _notify   : implementação interna da notificação.
        """
        self.value = value
        self._notify()

    def subscribe(
        self,
        listener: Listener,
        *,
        call_immediately: bool = False,
    ) -> ReactiveNode:
        """
        Registra um listener para reagir às mudanças deste nó.

        O listener é adicionado ao final de :attr:`listeners` e será chamado
        em cada invocação de :meth:`set`.

        Para tornar um widget reativo, passe uma função que atualize o widget::

            progress = widgets.FloatProgress()
            node.subscribe(lambda v: setattr(progress, "value", v))

        Parameters
        ----------
        listener : Listener
            Callable ``(value: Any) -> None`` a ser registrado.
        call_immediately : bool, optional
            Se ``True``, o listener é chamado imediatamente com o valor atual
            logo após ser registrado.  Útil para inicializar a interface sem
            precisar forçar um :meth:`set` manual.  Padrão: ``False``.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento de chamadas::

                node.subscribe(fn_a).subscribe(fn_b)

        See Also
        --------
        unsubscribe : remove um listener previamente registrado.
        to_widget   : atalho que cria e registra um listener para um widget.
        """
        self.listeners.append(listener)
        if call_immediately:
            listener(self.value)
        return self

    def unsubscribe(self, listener: Listener) -> None:
        """
        Remove um listener previamente registrado.

        Se o listener não estiver na lista, a chamada é ignorada silenciosamente.

        Parameters
        ----------
        listener : Listener
            O mesmo callable passado a :meth:`subscribe`.
        """
        if listener in self.listeners:
            self.listeners.remove(listener)

    def _notify(self) -> None:
        """
        Invoca todos os listeners com o valor atual do nó.

        A lista de listeners é copiada antes da iteração para permitir que um
        listener se cancele (via :meth:`unsubscribe`) sem causar erros de
        modificação concorrente.

        Notes
        -----
        Método de uso interno; prefira :meth:`set` para acionar a propagação.
        """
        for listener in list(self.listeners):
            listener(self.value)

    def map(
        self,
        func: Callable[[Any], Any],
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Cria um nó filho derivado deste, transformando seu valor.

        Internamente, registra um listener em ``self`` que chama :meth:`set`
        no nó filho sempre que o valor pai mudar.

        Parameters
        ----------
        func : Callable[[Any], Any]
            Função de transformação unária aplicada ao valor deste nó.
        initialize : bool, optional
            Se ``True``, o nó filho é inicializado imediatamente com
            ``func(self.value)``.  Padrão: ``True``.

        Returns
        -------
        ReactiveNode
            Novo nó derivado.  Qualquer listener registrado nele será
            notificado quando *este* nó mudar.

        Examples
        --------
        >>> x = ReactiveNode(3)
        >>> x_sq = x.map(lambda v: v ** 2)
        >>> x_sq.get()
        9
        >>> x.set(5)
        >>> x_sq.get()
        25

        See Also
        --------
        computed : versão que aceita múltiplos nós de entrada.
        """
        initial = func(self.value) if initialize else None
        child = ReactiveNode(initial)

        def listener(value: Any) -> None:
            child.set(func(value))

        self.subscribe(listener)
        return child

    def to_widget(
        self,
        widget: widgets.Widget,
        attr: str,
        transform: Callable[[Any], Any] = identity,
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Liga este nó a um atributo de um widget, tornando-o reativo.

        Este método encapsula o padrão manual de registrar um listener que
        atualiza um widget::

            # Equivalente explícito ao que to_widget faz internamente:
            node.subscribe(lambda v: setattr(widget, attr, transform(v)),
                           call_immediately=initialize)

        Parameters
        ----------
        widget : widgets.Widget
            Widget de destino a ser atualizado.
        attr : str
            Nome do atributo do widget a ser escrito, por exemplo ``"value"``,
            ``"description"`` ou ``"style"``.
        transform : Callable[[Any], Any], optional
            Transformação aplicada ao valor do nó antes da atribuição.
            Padrão: :func:`identity` (nenhuma transformação).
        initialize : bool, optional
            Se ``True``, o atributo do widget é definido imediatamente com o
            valor atual do nó (sem necessidade de um :meth:`set` inicial).
            Padrão: ``True``.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento::

                node.to_widget(w1, "value").to_widget(w2, "description", str)

        Examples
        --------
        Exibir o quadrado do valor de um slider em uma barra de progresso::

            x_sq.to_widget(progress, "value")

        Exibir o valor como texto formatado em um label::

            x.to_widget(label, "value", transform=lambda v: f"x = {v:.2f}")

        See Also
        --------
        subscribe : método de baixo nível para registrar listeners manualmente.
        """
        def listener(value: Any) -> None:
            setattr(widget, attr, transform(value))

        self.subscribe(listener, call_immediately=initialize)
        return self


class WidgetNode(ReactiveNode):
    """
    Nó reativo cuja *fonte* é um atributo observável de um widget ipywidgets.

    :class:`WidgetNode` é a ponte de entrada do sistema reativo: ele observa
    um widget (tipicamente um controle interativo como um slider) e propaga
    as mudanças do usuário para os listeners registrados, que por sua vez podem
    atualizar outros widgets ou recalcular valores derivados.

    Internamente, o nó usa :meth:`widgets.Widget.observe` para escutar
    alterações no atributo indicado e chama :meth:`ReactiveNode.set` sempre
    que o widget muda — o que aciona toda a cadeia de listeners cadastrados.

    Parameters
    ----------
    widget : widgets.Widget
        Widget de origem cujos valores serão observados.
    attr : str
        Nome do atributo observado, normalmente ``"value"``.

    Examples
    --------
    Criar um nó a partir de um slider e registrar um listener::

        slider = widgets.FloatSlider(min=0, max=10, value=5)
        node = WidgetNode(slider, "value")
        node.subscribe(lambda v: print(f"slider movido para {v}"))

    Na prática, use a função :func:`bind` como atalho::

        node = bind(slider)

    See Also
    --------
    bind : função fábrica de conveniência para criar um :class:`WidgetNode`.
    """

    def __init__(self, widget: widgets.Widget, attr: str) -> None:
        self.widget = widget
        self.attr = attr
        self._widget_handler = self._make_handler()
        super().__init__(getattr(widget, attr))
        widget.observe(self._widget_handler, names=attr)

    def _make_handler(self) -> Callable[[dict[str, Any]], None]:
        """
        Cria o callback interno que traduz eventos do ipywidgets para :meth:`set`.

        O ipywidgets entrega um dicionário ``change`` com chaves como ``"new"``
        e ``"old"``; este handler extrai ``change["new"]`` e o passa a
        :meth:`ReactiveNode.set`, disparando todos os listeners do nó.

        Returns
        -------
        Callable[[dict[str, Any]], None]
            Função compatível com a assinatura esperada por
            :meth:`widgets.Widget.observe`.
        """
        def handler(change: dict[str, Any]) -> None:
            self.set(change["new"])

        return handler

    def unlink(self) -> None:
        """
        Remove a observação do widget de origem, desativando o nó.

        Após chamar este método, mudanças no widget não propagarão mais
        atualizações pelos listeners.  Útil para evitar vazamentos de memória
        ao descartar um painel interativo.
        """
        self.widget.unobserve(self._widget_handler, names=self.attr)


def bind(widget: widgets.Widget, attr: str = "value") -> WidgetNode:
    """
    Cria um :class:`WidgetNode` a partir de um atributo de um widget.

    Função fábrica de conveniência que evita a instanciação explícita de
    :class:`WidgetNode`.

    Parameters
    ----------
    widget : widgets.Widget
        Widget de origem a ser observado.
    attr : str, optional
        Atributo observado.  Padrão: ``"value"``, que corresponde ao valor
        principal da maioria dos widgets interativos (sliders, caixas de texto,
        etc.).

    Returns
    -------
    WidgetNode
        Nó reativo vinculado ao widget.

    Examples
    --------
    ::

        slider = widgets.FloatSlider(min=0, max=100, value=50)
        x = bind(slider)
        x.subscribe(lambda v: print(v))   # imprime cada vez que o slider muda

    See Also
    --------
    WidgetNode : classe subjacente criada por esta função.
    computed   : combina múltiplos nós em um único nó derivado.
    """
    return WidgetNode(widget, attr)


def computed(
    *nodes: ReactiveNode,
    func: Transform,
    initialize: bool = True,
) -> ReactiveNode:
    """
    Cria um nó derivado de um ou mais nós de entrada.

    Para cada nó de entrada é registrado um listener que recalcula o valor
    derivado chamando ``func`` com os valores atuais de *todas* as
    dependências.  Assim, qualquer mudança em qualquer dependência propaga-se
    automaticamente para o nó resultante e, por conseguinte, para os listeners
    registrados nele.

    Parameters
    ----------
    *nodes : ReactiveNode
        Um ou mais nós de entrada (dependências).
    func : Transform
        Função que combina os valores das dependências na ordem em que foram
        passados::

            resultado = func(nodes[0].get(), nodes[1].get(), ...)

    initialize : bool, optional
        Se ``True``, o nó derivado é inicializado imediatamente avaliando
        ``func`` com os valores atuais das dependências.  Padrão: ``True``.

    Returns
    -------
    ReactiveNode
        Novo nó cujo valor é sempre ``func(*[n.get() for n in nodes])``,
        atualizado toda vez que qualquer dependência mudar.

    Examples
    --------
    Nó que representa a soma de dois sliders::

        a = bind(widgets.IntSlider(value=1))
        b = bind(widgets.IntSlider(value=2))
        soma = computed(a, b, func=lambda x, y: x + y)
        soma.to_widget(label, "value", transform=str)

    See Also
    --------
    ReactiveNode.map : versão simplificada para um único nó de entrada.
    bind             : cria um nó de entrada a partir de um widget.
    """
    def evaluate() -> Any:
        return func(*(node.get() for node in nodes))

    result = ReactiveNode(evaluate() if initialize else None)

    def recompute(_: Any) -> None:
        result.set(evaluate())

    for node in nodes:
        node.subscribe(recompute)

    return result


In [ ]:
from pathlib import Path


class FuncaoTabelada:
    """Callable respaldado por uma tabela lida de arquivo."""

    def __init__(self, tabela: list, x_min: int, x_max: int):
        self._tabela = tabela
        self.x_min = x_min
        self.x_max = x_max

    def __call__(self, x: float | int):
        if not (self.x_min <= x <= self.x_max):
            raise ValueError(f"x={x} fora do intervalo [{self.x_min}, {self.x_max}]")
        return self._tabela[int(round(x)) - self.x_min]


class FuncaoAnalitica:
    """Callable respaldado por uma função Python."""

    DEFAULT_MAX = 10_000

    def __init__(self, func: Callable, x_min: int, x_max: int):
        self._func = func
        self.x_min = x_min
        self.x_max = x_max

    def __call__(self, x: int):
        if not (self.x_min <= x <= self.x_max):
            raise ValueError(f"x={x} fora do intervalo [{self.x_min}, {self.x_max}]")
        return self._func(x)


def fabrica(
    fonte: str | Callable,
    x_min: int | None = None,
    x_max: int | None = None,
) -> FuncaoTabelada | FuncaoAnalitica:
    """
    Cria um callable a partir de um arquivo ou de uma função.

    Parâmetros
    ----------
    fonte   : caminho de arquivo (str/Path) ou qualquer Callable
    x_min   : valor mínimo de x (padrão: 0)
    x_max   : valor máximo de x (padrão: nº de linhas do arquivo,
               ou FuncaoAnalitica.DEFAULT_MAX se fonte for função)
    """

    if callable(fonte):
        x_min = x_min if x_min is not None else 0
        x_max = x_max if x_max is not None else FuncaoAnalitica.DEFAULT_MAX
        return FuncaoAnalitica(fonte, x_min, x_max)

    # fonte é um arquivo
    caminho = Path(fonte)
    linhas = caminho.read_text().splitlines()
    tabela = [float(linha.strip()) for linha in linhas if linha.strip()]

    x_min = x_min if x_min is not None else 0
    x_max = x_max if x_max is not None else x_min + len(tabela) - 1

    if (x_max - x_min + 1) != len(tabela):
        raise ValueError(
            f"Intervalo [{x_min}, {x_max}] incompatível "
            f"com {len(tabela)} entradas na tabela."
        )

    return FuncaoTabelada(tabela, x_min, x_max)

In [ ]:
from IPython.display import display as ipython_display

%matplotlib widget

import matplotlib.pyplot as plt


"""
experiment_widget.py
====================
Widget interativo para exploração de funções e suas taxas de variação,
destinado ao material didático de Métodos Numéricos (UFRPE / DEINFO).

Depende da infraestrutura reativa do notebook (ReactiveNode, WidgetNode,
bind, computed) definida na célula anterior ao uso desta classe.

Uso básico
----------
::

    # No topo do notebook, antes de qualquer import de matplotlib:
    # %matplotlib widget        # necessário para o gráfico ser interativo

    exp = Experiment(
        func    = medicoes,     # FuncaoTabelada, FuncaoAnalitica ou Callable
        xmin    = 0,  xmax  = 19,
        ymin    = 0,  ymax  = 5000,
        label_x = "intervalo i  (t = i × 30 s)",
        label_y = "N (contagens)",
    )

    # Momentos didáticos progressivos — mesmo objeto, quatro chamadas:
    exp.display(show_y=True, show_diff=False, show_ratio=False, show_plot=False)
    exp.display(show_y=True, show_diff=True,  show_ratio=False, show_plot=False)
    exp.display(show_y=True, show_diff=True,  show_ratio=True,  show_plot=False)
    exp.display(show_y=True, show_diff=True,  show_ratio=True,  show_plot=True)

Parâmetros de display
---------------------
show_y     : barra de y = f(x)                       (padrão: True)
show_diff  : barra de Δy/Δx + segmento no gráfico    (padrão: True)
show_ratio : barra de (Δy/Δx)/y  ≈  −λ              (padrão: False)
show_plot  : gráfico matplotlib interativo            (padrão: False)

O slider de x é sempre exibido.

Nota sobre o backend matplotlib
--------------------------------
Com ``%matplotlib widget`` (ipympl), o gráfico responde ao slider em
tempo real: apenas os dados dos artistas existentes são atualizados,
sem recriar a figura.

Com ``%matplotlib inline``, o gráfico é estático — as barras de
progresso continuam reativas, mas o gráfico não se atualiza ao mover
o slider.
"""


# ─────────────────────────────────────────────────────────────────────────────
# Constantes visuais
# ─────────────────────────────────────────────────────────────────────────────

_BAR_STYLES = {
    "y":     "danger",   # vermelho — "o que existe agora"
    "diff":  "info",     # azul     — "como está mudando"
    "ratio": "warning",  # amarelo  — "a proporção constante"
}

_COLOR_DOTS    = "#e05252"   # cruzinhas — todos os pontos do dataset
_COLOR_MARKER  = "#1a6ebd"   # círculo   — ponto selecionado pelo slider
_COLOR_VLINE   = "#aaaaaa"   # linha vertical no x atual
_COLOR_TANGENT = "#f5a623"   # segmento de Δy/Δx

_SLIDER_WIDTH  = "90%"
_ROW_WIDTH     = "90%"
_LABEL_WIDTH   = "200px"
_LABEL_MARGIN  = "0 0 0 14px"


# ─────────────────────────────────────────────────────────────────────────────
# Classe principal
# ─────────────────────────────────────────────────────────────────────────────

class Experiment:
    """
    Widget Jupyter para exploração interativa de uma função e suas taxas.

    Parâmetros
    ----------
    func       : callable ``f(x) -> float`` — FuncaoTabelada, FuncaoAnalitica
                 ou qualquer Callable Python.
    xmin/xmax  : domínio do slider e do gráfico.
    ymin/ymax  : intervalo das barras de progresso.  ymax=None usa func(xmax).
    x0         : valor inicial do slider.  None usa xmin.
    step       : passo do slider.  Padrão: 1 (discreto).
    label_x    : rótulo do slider e do eixo x.
    label_y    : rótulo da barra de y e do eixo y.
    fig_width, fig_height : dimensões da figura em polegadas.
    """

    def __init__(
        self,
        func:       Callable,
        xmin:       float,
        xmax:       float,
        ymin:       float,
        ymax:       float | None = None,
        x0:         float | None = None,
        step:       float = 1,
        label_x:    str = "x",
        label_y:    str = "y",
        fig_width:  float = 7.0,
        fig_height: float = 3.5,
    ) -> None:
        self.func       = func
        self.xmin       = xmin
        self.xmax       = xmax
        self.ymin       = ymin
        self.ymax       = ymax if ymax is not None else func(xmax)
        self.x0         = x0   if x0   is not None else xmin
        self.step       = step
        self.label_x    = label_x
        self.label_y    = label_y
        self.fig_width  = fig_width
        self.fig_height = fig_height

        # Pré-calcular pontos do domínio para o gráfico
        self._xs: list[float] = []
        self._ys: list[float] = []
        x = xmin
        while x <= xmax + 1e-9:
            try:
                self._xs.append(x)
                self._ys.append(float(func(x)))
            except Exception:
                pass
            x = round(x + step, 10)

        self._build_widgets()
        self._build_plot()
        self._wire_reactivity()

    # ── Construção das barras de progresso ───────────────────────────────────

    def _build_widgets(self) -> None:
        self._slider = widgets.FloatSlider(
            min=self.xmin, max=self.xmax,
            step=self.step, value=self.x0,
            description=self.label_x,
            continuous_update=True,
            style={"description_width": "initial"},
            layout=widgets.Layout(width=_SLIDER_WIDTH),
        )

        # Barra y
        self._bar_y = widgets.FloatProgress(
            min=self.ymin, max=self.ymax,
            value=self._safe_call(self.x0),
            description=self.label_y,
            bar_style=_BAR_STYLES["y"],
            layout=widgets.Layout(flex="1 1 auto"),
        )
        self._lbl_y = widgets.Label(
            value=self._fmt_y(self.x0),
            layout=widgets.Layout(width=_LABEL_WIDTH, margin=_LABEL_MARGIN),
        )

        # Barra Δy/Δx
        self._bar_diff = widgets.FloatProgress(
            min=0, max=abs(self.ymax - self.ymin),
            value=abs(self._diff(self.x0)),
            description="Δy/Δx",
            bar_style=_BAR_STYLES["diff"],
            layout=widgets.Layout(flex="1 1 auto"),
        )
        self._lbl_diff = widgets.Label(
            value=self._fmt_diff(self.x0),
            layout=widgets.Layout(width=_LABEL_WIDTH, margin=_LABEL_MARGIN),
        )

        # Barra (Δy/Δx)/y
        ratio_max = abs(self._ratio(self.xmin)) * 1.5 or 1.0
        self._bar_ratio = widgets.FloatProgress(
            min=0, max=ratio_max,
            value=abs(self._ratio(self.x0)),
            description="(Δy/Δx)/y",
            bar_style=_BAR_STYLES["ratio"],
            layout=widgets.Layout(flex="1 1 auto"),
        )
        self._lbl_ratio = widgets.Label(
            value=self._fmt_ratio(self.x0),
            layout=widgets.Layout(width=_LABEL_WIDTH, margin=_LABEL_MARGIN),
        )

    # ── Construção da figura matplotlib ──────────────────────────────────────

    def _build_plot(self) -> None:
        """
        Cria figura e artistas uma única vez.

        Updates posteriores apenas chamam set_data() nos artistas existentes
        — sem recriar a figura — o que elimina o piscar ao mover o slider.
        """
        with plt.ioff():
            self._fig, self._ax = plt.subplots(
                figsize=(self.fig_width, self.fig_height)
            )

        ax = self._ax
        ax.set_xlabel(self.label_x, fontsize=10)
        ax.set_ylabel(self.label_y, fontsize=10)
        ax.set_xlim(self.xmin - 0.5 * self.step, self.xmax + 0.5 * self.step)
        ax.set_ylim(self.ymin, self.ymax * 1.08)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=9)

        # Todos os pontos do dataset — fixos, nunca recriados
        ax.plot(
            self._xs, self._ys,
            marker="x", markersize=7, markeredgewidth=2.0,
            color=_COLOR_DOTS, linestyle="none",
            label=self.label_y, zorder=3,
        )

        x0 = self.x0
        y0 = self._safe_call(x0)

        # Marcador do ponto selecionado (círculo azul)
        self._marker, = ax.plot(
            [x0], [y0],
            marker="o", markersize=9, markeredgewidth=1.5,
            color=_COLOR_MARKER, markerfacecolor="white",
            markeredgecolor=_COLOR_MARKER,
            zorder=5, label=f"{self.label_x} atual",
        )

        # Linha vertical pontilhada no x atual
        self._vline = ax.axvline(
            x=x0, color=_COLOR_VLINE,
            linewidth=1.2, linestyle=":", alpha=0.8,
        )

        # Segmento Δy/Δx (visível somente quando show_diff=True)
        x1 = min(x0 + self.step, self.xmax)
        y1 = self._safe_call(x1)
        self._tangent, = ax.plot(
            [x0, x1], [y0, y1],
            color=_COLOR_TANGENT, linewidth=2.5,
            solid_capstyle="round",
            label="Δy/Δx", zorder=4,
            visible=False,
        )

        # Anotação flutuante com o valor atual
        self._annotation = ax.annotate(
            f"  {self.label_y} = {y0:.1f}",
            xy=(x0, y0),
            xytext=(6, 6), textcoords="offset points",
            fontsize=8.5, color=_COLOR_MARKER,
            zorder=6,
        )

        self._fig.tight_layout(rect=(0.07, 0.0, 1.0, 1.0))  # margem esquerda para toolbar do ipympl

    # ── Fiação reativa ────────────────────────────────────────────────────────

    def _wire_reactivity(self) -> None:
        """
        Conecta o slider a todos os artistas e labels via bind() / WidgetNode.

        A infraestrutura reativa (bind, ReactiveNode, WidgetNode) é importada
        do namespace do notebook em tempo de execução — não precisa ser
        importada aqui.
        """
        rx = bind(self._slider)   # WidgetNode — fonte primária de todos os updates

        # Barras de progresso
        rx.to_widget(self._bar_y,     "value", transform=lambda v: self._safe_call(v))
        rx.to_widget(self._lbl_y,     "value", transform=self._fmt_y)
        rx.to_widget(self._bar_diff,  "value", transform=lambda v: abs(self._diff(v)))
        rx.to_widget(self._lbl_diff,  "value", transform=self._fmt_diff)
        rx.to_widget(self._bar_ratio, "value", transform=lambda v: abs(self._ratio(v)))
        rx.to_widget(self._lbl_ratio, "value", transform=self._fmt_ratio)

        # Gráfico: listener direto que modifica artistas matplotlib
        rx.subscribe(self._update_plot, call_immediately=False)

    def _update_plot(self, x: float) -> None:
        """Listener chamado a cada mudança do slider: atualiza artistas."""
        y  = self._safe_call(x)
        x1 = min(x + self.step, self.xmax)
        y1 = self._safe_call(x1)

        self._marker.set_data([x], [y])
        self._vline.set_xdata([x, x])
        self._tangent.set_data([x, x1], [y, y1])
        self._annotation.set_text(f"  {self.label_y} = {y:.1f}")
        self._annotation.set_position((x, y))

        try:
            self._fig.canvas.draw_idle()   # só funciona com backend widget
        except Exception:
            pass

    # ── Interface pública ────────────────────────────────────────────────────

    def display(
        self,
        show_y:     bool = True,
        show_diff:  bool = True,
        show_ratio: bool = False,
        show_plot:  bool = False,
    ) -> None:
        """
        Renderiza o widget no notebook.

        Parâmetros
        ----------
        show_y     : exibe a barra de y = f(x).
        show_diff  : exibe a barra de Δy/Δx e o segmento laranja no gráfico.
        show_ratio : exibe a barra de (Δy/Δx)/y.
        show_plot  : exibe o gráfico matplotlib interativo.
        """
        self._tangent.set_visible(show_diff)

        rows: list[widgets.Widget] = [self._slider]
        if show_y:
            rows.append(self._make_row(self._bar_y,    self._lbl_y))
        if show_diff:
            rows.append(self._make_row(self._bar_diff, self._lbl_diff))
        if show_ratio:
            rows.append(self._make_row(self._bar_ratio, self._lbl_ratio))

        controls = widgets.VBox(
            rows,
            layout=widgets.Layout(width="100%", align_items="flex-start"),
        )

        if show_plot:
            canvas_row = widgets.HBox(
                [self._fig.canvas],
                layout=widgets.Layout(width="100%", justify_content="center"),
            )
            ipython_display(widgets.VBox(
                [controls, canvas_row],
                layout=widgets.Layout(width="100%"),
            ))
        else:
            ipython_display(controls)

    # ── Cálculos numéricos ───────────────────────────────────────────────────

    def _safe_call(self, x: float) -> float:
        try:
            return float(self.func(x))
        except Exception:
            return 0.0

    def _diff(self, x: float) -> float:
        """Δy/Δx — diferença progressiva; regressiva no último ponto."""
        try:
            if x + self.step <= self.xmax:
                return (self.func(x + self.step) - self.func(x)) / self.step
            return (self.func(x) - self.func(x - self.step)) / self.step
        except Exception:
            return 0.0

    def _ratio(self, x: float) -> float:
        """(Δy/Δx) / y  ≈  −λ para decaimento exponencial."""
        try:
            y = self._safe_call(x)
            return 0.0 if y == 0 else self._diff(x) / y
        except Exception:
            return 0.0

    # ── Formatação de labels ─────────────────────────────────────────────────

    def _fmt_y(self, x: float) -> str:
        try:
            return f"{self.label_y} = {self._safe_call(x):.2f}"
        except Exception:
            return f"{self.label_y} = —"

    def _fmt_diff(self, x: float) -> str:
        try:
            return f"Δy/Δx = {self._diff(x):+.4f}"
        except Exception:
            return "Δy/Δx = —"

    def _fmt_ratio(self, x: float) -> str:
        try:
            return f"(Δy/Δx)/y = {self._ratio(x):+.5f}"
        except Exception:
            return "(Δy/Δx)/y = —"

    # ── Helper de layout ─────────────────────────────────────────────────────

    @staticmethod
    def _make_row(bar: widgets.Widget, label: widgets.Widget) -> widgets.HBox:
        return widgets.HBox(
            [bar, label],
            layout=widgets.Layout(width=_ROW_WIDTH, align_items="center"),
        )


In [ ]:
medicoes = fabrica("datasets/ba137m_contagens_clean.txt", x_min=0, x_max=19)

exp = Experiment(
    func    = medicoes,
    xmin    = 0,   xmax  = 19,
    ymin    = 0,   ymax  = 5000,
    step    = 1,
    label_x = "intervalo i  (t = i × 30 s)",
    label_y = "N (contagens)",
)

# Aluno compara N e ΔN/Δt movendo o slider.
# O gráfico aparece aqui pela primeira vez, com o segmento laranja de Δy/Δx.
# Pergunta: quando N é grande, ΔN/Δt é grande ou pequeno?
exp.display(show_y=True, show_diff=True, show_ratio=False, show_plot=True)

In [ ]:
def experiment(xmin: Number, xmax: Number, ymin: Number, ymax: Number | None = None, x0: Number = 0, y0: Number = 0, func: Callable = lambda x, x0 = 0, y0 = 0: x, *, discrete=True):

    ymax = ymax if ymax is not None else func(xmax)
    step = 1 if discrete else 0.1

    xslider = widgets.FloatSlider(min=xmin, max=xmax, step=step, value=x0, description="x")
    yprogress = widgets.FloatProgress(min=ymin, max=ymax, value = y0, description="y", bar_style="danger")
    diffprogress = widgets.FloatProgress(min=0, max=ymax, value = 0, description="Δy/Δx")
    y_label = widgets.Label()
    diff_label = widgets.Label(value=f"step = {step}")
    xslider.layout = widgets.Layout(width="90%")
    yprogress.layout = widgets.Layout(flex="1 1 auto")
    diffprogress.layout = widgets.Layout(flex="1 1 auto")
    y_label.layout = widgets.Layout(width="180px", margin="0 0 0 12px")
    diff_label.layout = widgets.Layout(width="180px", margin="0 0 0 12px")

    xslider_reactive = bind(xslider)
    diffprogress_reactive = bind(diffprogress)
    xslider_reactive.to_widget(yprogress, "value", transform=lambda v: func(v, x0, y0))
    xslider_reactive.to_widget(diffprogress, "value", transform=lambda v: (func(v + step, x0, y0) - func(v, x0, y0)) / step)
    xslider_reactive.to_widget(y_label, "value", transform=lambda v: f"y = {func(v, x0, y0):.2f}")
    diffprogress_reactive.to_widget(diff_label, "value", transform=lambda v: f"dy/dx ≈ {v:.2f}")

    y_row = widgets.HBox(
        [yprogress, y_label],
        layout=widgets.Layout(width="90%", align_items="center")
    )
    diff_row = widgets.HBox(
        [diffprogress, diff_label],
        layout=widgets.Layout(width="90%", align_items="center")
    )
    controls = widgets.VBox(
        [xslider, y_row, diff_row],
        layout=widgets.Layout(width="100%", align_items="flex-start")
    )
    display(controls)



In [ ]:
# Crescimento populacional exponencial: y = y0 * 2^x

yinit = 100
xmax = 5
x0 = 0
y0 = yinit
func = lambda x, x0, y0: y0 * 2**x

experiment(0, xmax, 0, yinit * 2**xmax, x0 = x0, y0 = y0, func=func)

In [ ]:
tmin = 0
tmax = 5
pinit = 100


t = widgets.IntSlider(min=tmin, max=tmax, step=1, value=tmin, description="t")
p = widgets.FloatProgress(min=0, max=pinit*2**tmax, value = pinit, description="2^t", bar_style="danger")
p_value = widgets.Label()
p.layout.width = "1000px"

t_reactive = bind(t)
p_reactive = bind(p)
t_reactive.to_widget(p, "value", transform=lambda v: pinit*2**v)
p_reactive.to_widget(p_value, "value", transform=str)

display(t)
display(p)
display(p_value)